# Convolutional Neural Networks - how a network looks at a picture

## Important Information

- Email: [joanna_bieri@redlands.edu](mailto:joanna_bieri@redlands.edu)
- Office Hours take place in Duke 209, [Office Hours Schedule](https://joannabieri.com/schedule.html)
- [Class Website](https://joannabieri.com/machine_learning.html)
- [Syllabus](https://joannabieri.com/machinelearning/IntroMachineLearning.pdf)

:::{.callout-important icon=false}
## How to use these notes

Two kinds of box show up in these notes.

**Blue Q boxes** are questions for you to answer **by hand, in a notebook, with a pen.** Not because I am old fashioned. Writing something down by hand is slow, and slow is the point: it is very hard to write an explanation you do not actually understand. You are welcome to use AI in this class for the mechanics of code, but these boxes are the part where you do the thinking yourself. Bring your written notes to class, I will ask to see them.

**Green You Try boxes** are optional code for you to work through. Nothing is collected and nothing is graded. They are there because you will learn more from changing a number and rerunning than from watching me do it.

**Every code cell begins with a tag** that tells you what to do with it.

- `# RUN THIS.` Setup, loading data, a plot. Copy it, run it, move on. You do not need to be able to write it from memory.
- `# LEARN TO WRITE THIS.` The pattern of the day. The homework will ask you for it, and so will the exam. Type it out yourself at least once rather than pasting it.
- `# DEMO ONLY.` Code that exists to show one idea. You do not need to be able to write it yourself.
- `# UNDERSTAND THESE STEPS.` Code that shows how something works inside, one small step at a time. Read it and make sure you understand what each line does. You do not need to memorize it or be able to write it from scratch.
:::

**Reading:** Geron chapter 12, the sections **Convolutional Layers**, **Pooling Layers** and **CNN Architectures**. In CNN Architectures, read LeNet-5 carefully and skim the rest.

Every network we have built so far took in one row of a table: 8 numbers about a neighborhood, 11 numbers about a wine. Today the data is **pictures**.

A picture is a grid of numbers, one number per pixel. We could flatten the grid into one long row and hand it to any of the neural networks we have been training. Each pixel is a variable flowing into the network and we train the network to learn what is in the picture. That can work, but it throws away something important, **which pixels are next to each other**. We completely lose information about neighboring pixels! A **convolutional neural network (CNN)** keeps that information. It looks at the picture through a small window, one patch at a time. 

Here is the plan for today:

1. Load a data set of small pictures of clothes.
2. Train an MLP on the flattened pictures, so we have a number to beat.
3. Learn what a **convolution** is, with a picture and one small example.
4. Learn what **pooling** is.
5. Put them together into a small CNN and train it.
6. See what this looks like in real life, in one cell.

# The Data - Fashion MNIST, Small Pictures of Clothes

**Fashion MNIST** is 70,000 black and white pictures of clothes, each one 28 by 28 pixels, in 10 classes (T-shirt, trouser, sneaker, bag, ...). It is the data set the reading uses. `torchvision` downloads it for us, about 30 MB the first time, and saves it in the `data` folder, so after that it loads right away. 

To keep training fast on a laptop we only use the first 12,000 training pictures: 10,000 to train on and 2,000 to validate on. The separate test set we leave alone.

In [ ]:
# RUN THIS. Imports, and the Fashion MNIST pictures as tensors.
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
from torch.utils.data import TensorDataset, DataLoader

fashion = torchvision.datasets.FashionMNIST(
    root="data",              # the folder to save the pictures in
    train=True,               # the 60,000 training pictures (train=False gives the test set)
    download=True)            # download them the first time

class_names = fashion.classes                # the names of the 10 kinds of clothes

# The pixels are whole numbers from 0 to 255. Divide by 255 so they go from 0 to 1.
images = fashion.data[:12000].float() / 255
labels = fashion.targets[:12000]

# A CNN wants the pictures as (number of pictures, channels, height, width).
# Black and white pictures have 1 channel. A color picture would have 3 (red, green, blue).
images = images.reshape(-1, 1, 28, 28)

X_train = images[:10000]
y_train = labels[:10000]
X_valid = images[10000:]
y_valid = labels[10000:]
train_data = TensorDataset(X_train, y_train)  # pair the pictures with the labels, so a DataLoader can make batches

print("training pictures:  ", X_train.shape)
print("validation pictures:", X_valid.shape)
print("the 10 classes:", class_names)

Here we see what is in the data set. It is a bunch of pictures of clothes with labels of what the clothes are. We want to train a neural network to "look" at the picture and predict what type of clothing it is.

In [ ]:
# RUN THIS. Look at the first 10 training pictures.
fig, axes = plt.subplots(2, 5, figsize=(6, 2))
for i in range(10):
    ax = axes.flatten()[i]
    ax.imshow(X_train[i, 0], cmap="gray_r")      # [i, 0] is picture i, channel 0
    ax.set_title(class_names[y_train[i]])
    ax.axis("off")
plt.tight_layout()
plt.show()

print(f'The shape of each picture is {list(X_train[0,0].shape)} or a total of {X_train[0,0].shape[0]*X_train[0,0].shape[0]} pixels')

# Picking One of 10 Classes - Scores and a New Loss Function

So far our networks predicted one number, like a house value, and we used the mean squared error. Now the answer is one of 10 classes, so the network ends with a layer of **10 outputs**, one **score** for each kind of clothing. The class with the biggest score is the prediction. We can think of this as how certain the network is that the picture belongs to one of the classes.

The loss for this is `nn.CrossEntropyLoss()`. You hand it the 10 scores and the right answer, and the loss is small when the right class has a big score compared to the others. It is the 10 class version of the log loss from logistic regression on Day 9.

Let's look at one picture to see how this works. The first training picture is an **ankle boot** (class 9). Say the network gives it these 10 scores (made up numbers):

| class | name | score | probability | real answer |
|---|---|---|---|---|
| 0 | T-shirt/top | 0.5 | 0.038 | 0 |
| 1 | Trouser | -1.2 | 0.007 | 0 |
| 2 | Pullover | 0.3 | 0.031 | 0 |
| 3 | Dress | 0.1 | 0.026 | 0 |
| 4 | Coat | -0.4 | 0.016 | 0 |
| 5 | Sandal | 1.8 | 0.141 | 0 |
| 6 | Shirt | 0.2 | 0.028 | 0 |
| 7 | Sneaker | 2.1 | 0.190 | 0 |
| 8 | Bag | 0.9 | 0.057 | 0 |
| **9** | **Ankle boot** | **3.0** | **0.466** | **1** |

**The prediction.** The biggest score is 3.0, in spot 9, so the network predicts ankle boot. In PyTorch, `scores.argmax(dim=1)` finds the position of the biggest score for every picture. It is right!

**The real answer.** PyTorch stores the real answer in `y_train` as the class number, here 9. You can think of it as the last column: a row of 10 zeros with a single 1 in the right spot. This is called **one hot** encoding.

**Step 1 - scores to probabilities.** The scores get turned into probabilities that add up to 1. This is called **softmax**:

$$p_k = \frac{e^{\text{score}_k}}{e^{\text{score}_0} + e^{\text{score}_1} + \dots + e^{\text{score}_9}}$$

For the ankle boot: $p_9 = \dfrac{e^{3.0}}{43.06} = \dfrac{20.09}{43.06} = 0.466$

**Step 2 - the loss.** The loss only looks at the probability for the right class and takes minus the log of it, just like the log loss on Day 9:

$$\text{loss} = -\log(p_{\text{right class}}) = -\log(0.466) = 0.763$$

* If the right class gets a probability close to 1, the loss is close to 0.
* If the right class gets a small probability, the loss is big.

`nn.CrossEntropyLoss()` does both steps for us. The network learns by making this number small, which pushes the score for the right class up and the others down.

To see how we are doing we use **accuracy**, the fraction of pictures the network gets right.

Here is our training loop for today. A lot of it should look familiar with just a few changes.

In [ ]:
# RUN THIS. The mini-batch training loop, for 10 classes. It returns the validation accuracy after every epoch.
def train_model(model, n_epochs):
    torch.manual_seed(42)                         # so the batches come in the same order every time
    train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
    loss_fn = nn.CrossEntropyLoss()               # NEW LOSS FUNCTION!
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    valid_accuracy = []
    for epoch in range(n_epochs):
        model.train()
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            scores = model(X_batch)                # 10 scores for every picture in the batch - really the same as y_pred
            loss = loss_fn(scores, y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            scores = model(X_valid)
            y_pred = scores.argmax(dim=1)          # NEW WE FORCE THE MODEL TO CHOOSE THE HIGHEST SCORE
            accuracy = (y_pred == y_valid).float().mean().item() # We look at the accuracy because it is easy for humans to understand
        valid_accuracy.append(accuracy)
        print("epoch", epoch + 1, "  valid accuracy:", round(accuracy, 4))

    return valid_accuracy

# A Number to Beat - An MLP on Flattened Pictures

First the network we already know! Fully connected neural network with one hidden layer.

Each picture is 784 pixels so we use `nn.Flatten()` to turn each 28 by 28 picture into one row of 784 numbers. Then we send it to 100 hidden neurons, followed by ReLU(), and finally an output of 10 neurons that represents the score for each of the 10 possible categories.

In [ ]:
# RUN THIS. An MLP on the flattened pictures.
torch.manual_seed(42)
mlp = nn.Sequential(
    nn.Flatten(),                 # 1 x 28 x 28 picture -> a row of 784 numbers
    nn.Linear(784, 100),
    nn.ReLU(),
    nn.Linear(100, 10))           # 10 scores, one for each class

valid_accuracy_mlp = train_model(mlp, n_epochs=5)

After 5 epochs the MLP gets **0.858** of the validation pictures right. That is not bad for a network that has no idea it is looking at a picture!

## What the MLP throws away

When we flatten, pixels that are neighbors in the picture end up far apart in the row. The MLP has to learn from scratch that pixel 264 and pixel 292 are right on top of each other.

![Flattening keeps every pixel but loses which pixels are next to each other.](images/01-flatten.png){width=65%}

:::{.callout-note icon=false}
## Q1. Write this one out by hand

**a.** In your own words, what information does an MLP lose when we flatten a picture into one long row?

**b.** The first layer of our MLP is `nn.Linear(784, 100)`. How many weights and biases does it have to learn?
:::

---

# Convolution - A Small Window That Slides Over the Picture

A **convolution** looks at the picture through a bunch of small windows, usually 3 by 3 pixels. The window holds a grid of numbers called a **filter** (the reading also calls it a **kernel**). At every spot we multiply each pixel by the filter number on top of it and add everything up. The filter is something that the network learns during training. Doing this gives one number that represents the pixels in the square. Then the window slides over and we do it again.

Here is a tiny 5 by 5 picture: small numbers on the left (0 = white) and big numbers on the right (9 = dark gray). Here we made up numbers for the filter just for a demonstration!

![One step of a convolution. The output has one number for every place the window fits.](images/02-convolution.png)

This example filter gives a big number wherever the pixel values jump from small to big, left to right, and 0 where the picture is all the same. So it finds **vertical edges**. The neural network will learn what types of filters it needs, this is just an example. The grid of outputs is called a **feature map**: a map of where in the picture the filter found its feature.

Let's try this example filter on a real picture. Just run the code below, we will learn more about conv2d later!

In [ ]:
# DEMO ONLY. The vertical edge filter from the picture, on one Fashion MNIST picture.
edge_filter = torch.tensor([[-1.0, 0.0, 1.0],
                            [-1.0, 0.0, 1.0],
                            [-1.0, 0.0, 1.0]])
edge_filter = edge_filter.reshape(1, 1, 3, 3)          # (filters, channels, height, width)

picture = X_train[0:1]                                   # the first picture, kept in the 4 dimension shape
feature_map = nn.functional.conv2d(picture, edge_filter) # slide the filter over the picture

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(picture[0, 0], cmap="gray_r")
axes[0].set_title("the picture")
axes[1].imshow(feature_map[0, 0], cmap="RdBu")
axes[1].set_title("the feature map")
for ax in axes:
    ax.axis("off")
plt.show()

Notice when we plot the feature map it is blue along the left side of the boot, where the pixel values jump up, and red along the right side, where they drop back down. Everywhere else it is close to 0 (white). The filter found the boot's vertical edges.

Here is the big idea: **in a CNN we do not pick the filter numbers.** They are weights, and the network learns them with gradient descent, the same way it learns the weights in an `nn.Linear` layer. One filter might learn to find vertical edges, another horizontal edges, another a corner. The same 9 filter numbers are used at every spot in the picture, so a filter that finds an edge in the top corner also finds it in the bottom corner.

:::{.callout-note icon=false}
## Q2. Write this one out by hand

**a.** Use the filter from the picture, $\begin{bmatrix} -1 & 0 & 1 \\ -1 & 0 & 1 \\ -1 & 0 & 1 \end{bmatrix}$, on this 3 by 3 patch: $\begin{bmatrix} 2 & 2 & 8 \\ 2 & 2 & 8 \\ 2 & 2 & 8 \end{bmatrix}$. Multiply each pair and add them up. What number comes out?

**b.** What does this filter give for a patch where every pixel is the same number? Why does that make it an edge finder?
:::

---

# nn.Conv2d - Many Filters, All Learned

In PyTorch a convolution layer is `nn.Conv2d`. One convolutional layer holds many filters (out_channels), and each filter makes its own feature map, finding some different aspect of the image. Here are the important pieces:

* in_channels=1
    - This is how many numbers it takes to describe the color of one pixel.
    - It is 1 if we are dealing in grayscale (each pixel is a single number between 0 and 255 but we often scale the picture by dividing by 255)
    - It is 3 if we are dealing in RGB (we need a number for each of Red, Blue, and Green)
    - It can be 4 if you add RGB and Transparency
    - It can be even higher for things like satellite imagery!
    - In a second `nn.Conv2d` layer, `in_channels` is the `out_channels` of the layer before it.
  
* out_channels=16
    - This is the number of filters you are learning. 16 means we will have 16 feature maps that come out of this layer.
      
* kernel_size=3
    - This is the window size of the filter. Here we are setting it to be 3x3.

* padding=1
    - This helps us to decide what to do at the edges of the picture.
    - Here we are adding zeros around the edges so that the output after convolution stays shape 28x28.


In [ ]:
# UNDERSTAND THESE STEPS. One convolution layer, and the shape of what comes out.
torch.manual_seed(42)
conv = nn.Conv2d(
    in_channels=1,        # our pictures have 1 channel (black and white)
    out_channels=16,      # 16 filters, so 16 feature maps come out
    kernel_size=3,        # each filter is 3 by 3
    padding=1)            # add a border of zeros so the output stays 28 by 28

five_pictures = X_train[:5]
feature_maps = conv(five_pictures)

print("going in: ", five_pictures.shape)
print("coming out:", feature_maps.shape)
print("filter weights:", conv.weight.shape)
print("numbers to learn:", conv.weight.numel() + conv.bias.numel())

Five pictures of size 28x28 went in with 1 channel each [5, 1, 28, 28], and five pictures came out with **16 channels** representing each filter applied [5, 16, 28, 28], one feature map per filter. Without `padding=1` the 3 by 3 window only fits 26 times across a 28 pixel row, so the output would shrink to 26 by 26.

The whole layer has only **160** numbers to learn: 16 filters with 9 weights each, plus 16 biases. Compare that to the 78,500 in the first layer of the MLP. The small number is because each filter is reused at every spot in the picture.

---

# Pooling - Shrink the Picture, Keep the Strongest Signal

After a convolution we usually **pool**. Pooling is applying some sort of aggregation to a data window. We might average or take the max value. There are a few reasons for pooling:

* It reduces the number of things we need to compute! When you do a convolution it creates 16 feature maps each with 784 pixels. That is a lot of data!
* Pooling helps the network pay attention to information across a block of data. For example we just take the max so it looks at the biggest number.
* It keeps just the strongest signal found by the convolutional filters.

In our class we will use `nn.MaxPool2d(2)`, which looks at each 2 by 2 block of a feature map and keeps only the biggest number.

![Max pooling with a 2 by 2 window keeps the biggest number from each block.](images/03-pooling.png)

That cuts the height and the width of the image in half, so there is a quarter as much to compute in the next layer. It also means that if a feature moves over by a pixel, the result barely changes. A sneaker is a sneaker even if it sits a little to the left. A pooling layer has **nothing to learn**, it just does an aggregation - takes the biggest number.

In [ ]:
# UNDERSTAND THESE STEPS. Pooling cuts the height and width in half.
pool = nn.MaxPool2d(2)                  # keep the biggest number in every 2 by 2 block
pooled = pool(feature_maps)

print("before pooling:", feature_maps.shape)
print("after pooling: ", pooled.shape)

Make sure to keep track of the size changes of these tensors that we are creating. We are now down to five pictures of size 14x14 with 16 channels each [5, 16, 14, 14].

:::{.callout-note icon=false}
## Q3. Write this one out by hand

**a.** Do 2 by 2 max pooling by hand on $\begin{bmatrix} 5 & 1 & 0 & 2 \\ 3 & 2 & 4 & 1 \\ 0 & 0 & 1 & 6 \\ 2 & 8 & 3 & 1 \end{bmatrix}$.

**b.** A 28 by 28 picture with 1 channel goes through `nn.Conv2d(1, 8, kernel_size=3, padding=1)` and then `nn.MaxPool2d(2)`. What is the shape that comes out (channels, height, width)?
:::

---

# A Small CNN - Convolution, ReLU, Pooling, Twice, Then Decide

A CNN stacks these pieces into one network. 

* The first convolution layer finds simple things like edges. Then we do ReLU() followed by MaxPool2d(2)
* The second one works on the first one's feature maps, so it can find combinations of edges, like the curve of a collar or the sole of a shoe. Again we do ReLU() followed by MaxPool2d(2)
* At the end we flatten and use one `nn.Linear` layer to turn everything into 10 scores.
* We have to be careful here to calculate the correct number of neurons to accept all the data!

![The shape of the data after each layer of our CNN.](images/04-cnn-shapes.png)

**HERE IS OUR CODE FOR OUR FIRST CNN!**

In [ ]:
# LEARN TO WRITE THIS. A small CNN.
torch.manual_seed(42)
cnn = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1),    # 1 channel in, 16 feature maps out, 28 x 28
    nn.ReLU(),
    nn.MaxPool2d(2),                               # 16 x 14 x 14
    nn.Conv2d(16, 32, kernel_size=3, padding=1),   # 16 channels in, 32 feature maps out, 14 x 14
    nn.ReLU(),
    nn.MaxPool2d(2),                               # 32 x 7 x 7
    nn.Flatten(),                                  # a row of 32 * 7 * 7 = 1568 numbers
    nn.Linear(32 * 7 * 7, 10))                     # 10 scores, one for each class

print(cnn)

How many numbers do we need to learn?

In [ ]:
# RUN THIS. How many numbers does each network have to learn?
n_mlp = 0
for p in mlp.parameters():
    n_mlp = n_mlp + p.numel()

n_cnn = 0
for p in cnn.parameters():
    n_cnn = n_cnn + p.numel()

print("MLP:", n_mlp)
print("CNN:", n_cnn)

Notice that our Convolutional Neural Net (CNN) actually requires us to learn fewer parameters!

In [ ]:
# RUN THIS. Train the CNN the same way as the MLP.
valid_accuracy_cnn = train_model(cnn, n_epochs=5)

In [ ]:
# RUN THIS. The two learning curves.
epochs = range(1, 6)
plt.figure(figsize=(6, 4))
plt.plot(epochs, valid_accuracy_mlp, marker="o", label="MLP on flattened pictures")
plt.plot(epochs, valid_accuracy_cnn, marker="o", label="small CNN")
plt.xlabel("epoch")
plt.ylabel("validation accuracy")
plt.title("Fashion MNIST, 10,000 training pictures")
plt.legend()
plt.grid()
plt.show()

After 5 epochs the CNN gets **0.877** right, against **0.858** for the MLP, and it does that with **20,490** numbers to learn instead of **79,510**. Using the picture's shape really helps our network to learn. And this makes sense because the pixels surrounding a pixel give information about the photo.

It is slower though! A convolution does a lot of multiplying: every filter at every spot of every picture. So you will start to notice some lag during training, especially on older laptops.

:::{.callout-note icon=false}
## Q4. Write this one out by hand

**a.** The CNN has about a quarter as many numbers to learn as the MLP, and it does better. Explain how that can happen.

**b.** What does `nn.Flatten()` do in the CNN, and why does it have to come right before the last `nn.Linear` layer?
:::

---

# CNN Architectures - Bigger Versions of the Same Idea!

The CNNs people use for real are built from the same pieces we just used, just many more of them. Here are a few famous ones that you can learn more about in the reading:

- **LeNet-5** (1998) read handwritten digits for the US Postal Service. It is almost exactly our small CNN: two rounds of convolution and pooling, then a few `nn.Linear` layers.
- **AlexNet** (2012) won a big image contest, and won by a lot, starting the deep learning boom. Same pieces, more and bigger layers, trained on GPUs.
- **ResNet** (2015) has 34 to 152 layers. To train something that deep it uses two tricks: **skip connections**, which let the signal jump past a few layers, and **batch normalization**.

![Three famous CNNs as cartoons. Each box is a layer, and ResNet's curved arrows are the skip connections.](images/05-architectures.png){width=85%}

Remember batch normalization from Day 12? We said you would see it again inside CNNs, and here it is. In a big CNN there is an `nn.BatchNorm2d` right after almost every `nn.Conv2d`, keeping the signal a good size through all those layers. Our CNN has only two convolution layers, so it does not need it.

You will not train a ResNet from scratch on a laptop. In real projects people do the next best thing: they borrow a network that someone else already trained on millions of pictures, and fine-tune it for their own pictures. That is called **transfer learning**. There are sections of your book that talk about this if you are interested. It is a very powerful way to get a bigger network trained on a smaller machine!

:::{.callout-note icon=false}
## Q5. Write this one out by hand

**a.** In your own words, how does a CNN get from pixels to "this is a sneaker"? What do the first layers find, and what do the later layers find?

**b.** Why do very deep CNNs like ResNet use batch normalization, when our small CNN did fine without it?
:::

---

# Putting It Together

## What this looks like in real life - The Whole Process in One Cell

Here is everything in one cell, with no helper functions. The comments show the places you could change things.

In [ ]:
# LEARN TO WRITE THIS.
# A CNN from start to finish. It uses X_train, y_train, X_valid, y_valid from the loading cell.
torch.manual_seed(42)

# 1. The model. Here you could change the number of filters, add another round of
#    Conv2d, ReLU, MaxPool2d, or add nn.BatchNorm2d(16) right after the first Conv2d
model = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(16, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(32 * 7 * 7, 10))

# 2. The batches, the loss and the optimizer. Here you could change the batch size or the learning rate
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# 3. The training loop
n_epochs = 10
for epoch in range(n_epochs):
    model.train()                             # training mode
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()                 # clear the old gradients
        scores = model(X_batch)               # forward pass: 10 scores per picture
        loss = loss_fn(scores, y_batch)       # how far off are we?
        loss.backward()                       # backward pass
        optimizer.step()                      # one step of Adam

    # 4. After every epoch, check the validation accuracy
    model.eval()                              # prediction mode
    with torch.no_grad():
        y_pred = model(X_valid).argmax(dim=1)
        accuracy = (y_pred == y_valid).float().mean().item()
    print("epoch", epoch + 1, "  valid accuracy:", round(accuracy, 4))

Everything from today, in one list:

1. A picture is a grid of numbers. A black and white picture has 1 **channel**, a color picture has 3.
2. Flattening a picture for an MLP works, but it loses which pixels are next to each other.
3. A **convolution** slides a small **filter** over the picture, multiplying and adding, and makes a **feature map**. In a CNN the filter numbers are learned.
4. **Max pooling** keeps the biggest number in each 2 by 2 block. It halves the height and width and has nothing to learn.
5. A CNN stacks convolution, ReLU and pooling, then flattens and ends with an `nn.Linear` layer that gives one score per class.
6. For several classes, use `nn.CrossEntropyLoss()` and predict the class with the biggest score, `.argmax(dim=1)`.
7. Big CNNs use the same pieces, many more of them, with batch normalization to keep the signal a good size.

# New Commands Today

| Command | What it does | The thing that trips people up |
|---|---|---|
| `torchvision.datasets.FashionMNIST(root=, train=, download=True)` | downloads and loads Fashion MNIST | `.data` is the pixels (0 to 255), `.targets` is the labels |
| `images.reshape(-1, 1, 28, 28)` | puts the pictures in the shape a CNN wants | the order is (pictures, channels, height, width) |
| `nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)` | a convolution layer with `out_channels` filters | `in_channels` has to match the channels coming in |
| `nn.MaxPool2d(2)` | keeps the biggest number in each 2 by 2 block | it halves the height and width, it has nothing to learn |
| `nn.Flatten()` | turns each picture's feature maps into one row | the next `nn.Linear` needs the right size, eg. `32 * 7 * 7` |
| `nn.CrossEntropyLoss()` | the loss for picking one of several classes | hand it the scores, not the predicted classes |
| `scores.argmax(dim=1)` | the class with the biggest score, for every picture | `dim=1` means "across the 10 scores" |
| `nn.BatchNorm2d(n)` | batch normalization for `n` feature maps | goes right after an `nn.Conv2d` with `n` output channels |

:::{.callout-tip icon=false}
## You Try: optional code

Nothing here is collected. Work through it if you want the idea to stick.

**1.** In the real life cell, change the 16 and 32 filters to 8 and 16 (remember the `nn.Linear` size changes too). Is the accuracy much worse?

**2.** Add `nn.BatchNorm2d(16)` right after the first `nn.Conv2d` and `nn.BatchNorm2d(32)` right after the second. Does it change anything on this small CNN?

**3.** Look at the pictures the CNN gets wrong. Find a few where `y_pred` is not equal to `y_valid` and plot them with their real and predicted class names. Would you have gotten them right?
:::

# Before Next Class

1. In your lecture notes notebook, add your handwritten notes and answers to the questions.
2. Do the **Day 14 practice problems** in `HW_day14.ipynb`. They are practice, not graded, and the solutions are in the same folder.
3. **Weekly Homework 7** is due **Sunday 11/8 at 11:59pm**. It covers Day 13 and Day 14.
4. **Final project:** we take time for ideas at the end of class on Tuesday 11/3. Teams are due Tuesday 11/10. See the [Final Project Information](https://joannabieri.com/machinelearning/FinalProjectInfo.html) page.
5. Day 16 is Thursday 11/5: embeddings and sequences. Exam 2 is released after class that day.

# Answers to the Q Boxes

Try every one of these by hand first. These are short summaries, not full answers, and the writing out is the part that does the work.

:::{.callout-note collapse="true"}
## Q1. Flattening

**a.** Which pixels are next to each other. After flattening, two pixels that sit on top of each other in the picture end up 28 places apart in the row, and the MLP has to learn from scratch which pixels belong together.

**b.** $784 \times 100 = 78{,}400$ weights plus 100 biases, so 78,500 numbers.
:::

:::{.callout-note collapse="true"}
## Q2. Convolution

**a.** Each row is $2 \cdot (-1) + 2 \cdot 0 + 8 \cdot 1 = 6$, and three rows give $18$.

**b.** 0. The left column is multiplied by $-1$ and the right column by $+1$, so if they are the same number they cancel. The filter only gives a big number when the right side of the window is brighter than the left side, which is a vertical edge.
:::

:::{.callout-note collapse="true"}
## Q3. Pooling

**a.** $\begin{bmatrix} 5 & 4 \\ 8 & 6 \end{bmatrix}$

**b.** The convolution with `padding=1` keeps 28 by 28 and makes 8 channels, and the pooling halves the height and width: $8 \times 14 \times 14$.
:::

:::{.callout-note collapse="true"}
## Q4. The small CNN

**a.** Each filter has only 9 weights, and the same filter is reused at every spot in the picture, so the CNN needs far fewer numbers. And because it looks at small patches, it uses which pixels are neighbors, which the MLP has to figure out on its own.

**b.** It turns the 32 feature maps of size 7 by 7 into one row of 1568 numbers. `nn.Linear` takes a row of numbers, not a stack of grids, so the flattening has to happen right before it.
:::

:::{.callout-note collapse="true"}
## Q5. Architectures

**a.** The first convolution layer finds simple things like edges. The next layers combine those into bigger shapes, like a collar or a sole. The last `nn.Linear` layer turns those shapes into a score for each class, and the biggest score wins.

**b.** In a very deep network the signal can shrink or grow as it goes through all the layers (Day 11 and Day 12). Batch normalization keeps it a good size during training. With only two convolution layers that is not a problem.
:::